# TimesFM-3 on real data: 32M-point pre-training, zero-shot transfer, fine-tuning, calibration

This notebook takes the TimesFM-3 implementation in this repo through the full
foundation-model workflow, on CPU:

1. **A 32M-point real pre-training corpus** — seven public benchmark datasets
   (electricity, traffic, solar, ETTh1, ETTm1, ETTm2, exchange rates) spanning
   10-minute to daily frequencies, mixed 70/30 with the synthetic corpus.
2. **Corpus tricks that make a small budget productive** — multi-frequency
   augmentation (random-stride subsampling), *calendar covariates* fed through
   TimesFM-3's past-future covariate pathway, role randomization, and tempered
   source weighting so 862-series traffic doesn't drown out 7-series ETT.
3. **A true zero-shot claim** — **ETTh2 is held out entirely** (the dataset is
   never seen, not just its tail) and evaluated against last-value and
   seasonal-naive baselines.
4. **Few-shot fine-tuning** — the workflow real deployments follow: adapt the
   pre-trained checkpoint on the target dataset's own history and measure the
   gain over zero-shot on the same test windows.
5. **Covariate ablation and quantile calibration** — what the known-future
   pathway contributes, and whether the 9 quantiles mean what they claim.

Training the 5.2M-parameter `small` config takes ~45 minutes on CPU.

In [ ]:
import os, time
import numpy as np
import torch
import matplotlib.pyplot as plt

from timesfm3 import TimesFM3Config, TimesFM3Forecaster
from timesfm3.data import (
    MixedCorpus, RealWindowDataset, SyntheticMultivariateCorpus,
    calendar_covariates, load_csv_dataset,
)
from timesfm3.train import train

np.random.seed(0); torch.manual_seed(0)

DATA = "data/raw" if os.path.exists("data/raw") else "../data/raw"
STEPS = 10_000
BATCH = 16
CONTEXT_PATCHES, HORIZON_PATCHES = 8, 2
CONTEXT, HORIZON = 256, 64
CKPT = "timesfm3_real.pt"
CKPT_FT = "timesfm3_ett2_finetuned.pt"

## Datasets

| dataset | freq | channels | steps | points | role |
|---|---|---|---|---|---|
| traffic | hourly | 862 | 17,544 | 15.1M | train (first 80%) |
| electricity | hourly | 321 | 26,304 | 8.4M | train (first 80%) |
| solar | 10-min | 137 | 52,560 | 7.2M | train (first 80%) |
| ETTm1 / ETTm2 | 15-min | 7 each | 69,680 | 0.5M each | train (first 80%) |
| ETTh1 | hourly | 7 | 17,420 | 0.1M | train (first 80%) |
| exchange | daily | 8 | 7,588 | 0.06M | train (first 80%) |
| **ETTh2** | hourly | 7 | 17,420 | 0.1M | **held out — zero-shot / fine-tune eval** |

`periods` are each dataset's natural calendar cycles in steps (day, week); they
drive the calendar covariates. Fetch everything with `bash data/download.sh`.

In [ ]:
sources = {
    "traffic": load_csv_dataset(f"{DATA}/traffic.txt", "traffic", periods=(24, 168), skip_first_col=False),
    "electricity": load_csv_dataset(f"{DATA}/electricity.txt", "electricity", periods=(24, 168), skip_first_col=False),
    "solar": load_csv_dataset(f"{DATA}/solar_AL.txt", "solar", periods=(144, 1008), skip_first_col=False),
    "ETTh1": load_csv_dataset(f"{DATA}/ETTh1.csv", "ETTh1", periods=(24, 168)),
    "ETTm1": load_csv_dataset(f"{DATA}/ETTm1.csv", "ETTm1", periods=(96, 672)),
    "ETTm2": load_csv_dataset(f"{DATA}/ETTm2.csv", "ETTm2", periods=(96, 672)),
    "exchange": load_csv_dataset(
        f"{DATA}/exchange_rate.txt", "exchange", periods=(7,), skip_first_col=False
    ),
    "ETTh2": load_csv_dataset(f"{DATA}/ETTh2.csv", "ETTh2", periods=(24, 168)),
}
TRAIN_KEYS = ["traffic", "electricity", "solar", "ETTh1", "ETTm1", "ETTm2", "exchange"]
TRAIN_SOURCES = [sources[k] for k in TRAIN_KEYS]
HOLDOUT = sources["ETTh2"]

total = sum(s.values.size for s in TRAIN_SOURCES)
print(f"training corpus: {total/1e6:.1f}M real points across {len(TRAIN_SOURCES)} datasets")

fig, axes = plt.subplots(2, 2, figsize=(11, 5))
for ax, key in zip(axes.flat, ["traffic", "electricity", "solar", "ETTh1"]):
    ax.plot(sources[key].values[0, :600], lw=0.8)
    ax.set_title(f"{key} (channel 0)", fontsize=10)
fig.tight_layout(); plt.show()

## Pre-training

`RealWindowDataset` samples 320-step windows (256 context + 64 horizon):

- **Multi-frequency augmentation** — each window is subsampled with stride 1, 2 or 4,
  so hourly data also teaches 2-hourly/4-hourly dynamics and 10/15-min data teaches
  hourly-like dynamics.
- **Calendar covariates** — sin/cos of the dataset's daily and weekly cycles, computed
  from *absolute* positions (so strided windows keep true phase), appended as
  past-future covariates known arbitrarily far into the future.
- **Role randomization** — real channels randomly demoted to past-only covariates.
- **Tempered source weighting** — sources sampled ∝ points^0.5 so the corpus stays
  diverse instead of 50% traffic.

Real windows mix 70/30 with the synthetic corpus. Training uses the repo's standard
recipe: contiguous patch masking with randomized horizon length, Huber point loss +
pinball loss in normalized units, validation-based best-checkpoint saving.

In [ ]:
cfg = TimesFM3Config.small()
real = RealWindowDataset(
    cfg, TRAIN_SOURCES,
    context_patches=CONTEXT_PATCHES, horizon_patches=HORIZON_PATCHES, seed=11,
)
synth = SyntheticMultivariateCorpus(
    cfg, context_patches=CONTEXT_PATCHES, horizon_patches=HORIZON_PATCHES, seed=12
)
mixed = MixedCorpus(real, synth, primary_prob=0.7, seed=13)
val_real = RealWindowDataset(
    cfg, TRAIN_SOURCES,
    context_patches=CONTEXT_PATCHES, horizon_patches=HORIZON_PATCHES, seed=987_654,
)

history = []
t0 = time.time()
model = train(
    cfg, steps=STEPS, batch_size=BATCH,
    context_patches=CONTEXT_PATCHES, horizon_patches=HORIZON_PATCHES,
    dataset=mixed, val_dataset=val_real,
    checkpoint_path=CKPT, history=history, log_every=250, val_every=500,
)
print(f"wall time: {(time.time() - t0) / 60:.1f} min")

In [ ]:
tr = [(h["step"], h["train_loss"]) for h in history if "train_loss" in h]
va = [(h["step"], h["val_loss"]) for h in history if "val_loss" in h]
plt.figure(figsize=(8, 3.5))
plt.plot(*zip(*tr), label="train loss", lw=1)
plt.plot(*zip(*va), label="val loss (real windows)", marker="o", lw=1)
plt.xlabel("step"); plt.ylabel("Huber + pinball (normalized units)")
plt.legend(); plt.tight_layout(); plt.show()

## Zero-shot evaluation

Scaled MAE (per-series MAE / context std) over sliding windows, forecasting all
channels of a dataset **jointly** as multivariate targets:

- **in-domain, held-out time**: the last 20% of ETTh1,
- **zero-shot, held-out dataset**: ETTh2, never seen in any form,

each **with and without calendar covariates**, against last-value and
seasonal-naive (daily period) baselines.

In [ ]:
forecaster = TimesFM3Forecaster.from_checkpoint(CKPT)

def eval_windows(source, fc=None, num_windows=30, use_calendar=True, region=(0.8, 1.0)):
    fc = fc or forecaster
    vals = source.values
    n, t = vals.shape
    lo = int(t * region[0])
    hi = int(t * region[1]) - CONTEXT - HORIZON - 1
    starts = np.linspace(lo, hi, num_windows).astype(int)
    season = source.periods[0]
    reps = -(-HORIZON // season)
    errs = {"model": [], "last-value": [], "seasonal-naive": []}
    for s in starts:
        ctx = vals[:, s : s + CONTEXT]
        truth = vals[:, s + CONTEXT : s + CONTEXT + HORIZON]
        fcov = []
        if use_calendar:
            cal = calendar_covariates(int(s), CONTEXT + HORIZON, 1, source.periods)
            fcov = [cal[i] for i in range(len(cal))]
        res = fc.forecast(
            [ctx[i] for i in range(n)], HORIZON, future_covariates=fcov
        )
        for i in range(n):
            scale = max(ctx[i].std(), 1e-6)
            snaive = np.tile(ctx[i, -season:], reps)[:HORIZON]
            errs["model"].append(np.abs(res.point[i] - truth[i]).mean() / scale)
            errs["last-value"].append(np.abs(ctx[i, -1] - truth[i]).mean() / scale)
            errs["seasonal-naive"].append(np.abs(snaive - truth[i]).mean() / scale)
    return {k: float(np.mean(v)) for k, v in errs.items()}

rows = {
    "ETTh1 (in-domain, held-out time) + calendar": eval_windows(sources["ETTh1"], use_calendar=True),
    "ETTh1 (in-domain, held-out time), no calendar": eval_windows(sources["ETTh1"], use_calendar=False),
    "ETTh2 (ZERO-SHOT dataset) + calendar": eval_windows(HOLDOUT, use_calendar=True),
    "ETTh2 (ZERO-SHOT dataset), no calendar": eval_windows(HOLDOUT, use_calendar=False),
}
print(f"{'evaluation':<48}{'model':>8}{'last-val':>10}{'s-naive':>10}")
for name, r in rows.items():
    print(f"{name:<48}{r['model']:>8.3f}{r['last-value']:>10.3f}{r['seasonal-naive']:>10.3f}")

## Few-shot fine-tuning on the target dataset

Zero-shot is the headline capability, but deployments almost always have *some*
target history. We fine-tune the pre-trained checkpoint on windows from ETTh2's
**first 80% only** (the same region a practitioner would own; the test windows in
the last 20% stay untouched) — 600 steps at a low learning rate — and re-evaluate
on exactly the same test windows.

In [ ]:
ft_train = RealWindowDataset(
    cfg, [HOLDOUT],
    context_patches=CONTEXT_PATCHES, horizon_patches=HORIZON_PATCHES,
    train_fraction=0.8, seed=21,
)
ft_val = RealWindowDataset(
    cfg, [HOLDOUT],
    context_patches=CONTEXT_PATCHES, horizon_patches=HORIZON_PATCHES,
    train_fraction=0.8, seed=22,
)
base = TimesFM3Forecaster.from_checkpoint(CKPT)  # fresh copy of best weights
t0 = time.time()
train(
    cfg, steps=600, batch_size=BATCH, peak_lr=3e-5, warmup_steps=50,
    context_patches=CONTEXT_PATCHES, horizon_patches=HORIZON_PATCHES,
    dataset=ft_train, val_dataset=ft_val,
    checkpoint_path=CKPT_FT, log_every=200, val_every=100,
    model=base.model,
)
print(f"fine-tune wall time: {(time.time() - t0) / 60:.1f} min")

forecaster_ft = TimesFM3Forecaster.from_checkpoint(CKPT_FT)
ft = eval_windows(HOLDOUT, fc=forecaster_ft, use_calendar=True)
zs = rows["ETTh2 (ZERO-SHOT dataset) + calendar"]
print()
print(f"{'ETTh2 test windows':<32}{'model':>8}{'last-val':>10}{'s-naive':>10}")
print(f"{'zero-shot':<32}{zs['model']:>8.3f}{zs['last-value']:>10.3f}{zs['seasonal-naive']:>10.3f}")
print(f"{'fine-tuned (600 steps)':<32}{ft['model']:>8.3f}{ft['last-value']:>10.3f}{ft['seasonal-naive']:>10.3f}")

## Quantile calibration (zero-shot, ETTh2)

For each predicted quantile level q we measure the empirical fraction of
ground-truth values falling below the predicted quantile; a calibrated model
tracks the diagonal.

In [ ]:
def coverage(source, fc=None, num_windows=30, region=(0.8, 1.0)):
    fc = fc or forecaster
    vals = source.values
    n, t = vals.shape
    lo = int(t * region[0]); hi = int(t * region[1]) - CONTEXT - HORIZON - 1
    starts = np.linspace(lo, hi, num_windows).astype(int)
    below, count = np.zeros(9), 0
    for s in starts:
        ctx = vals[:, s : s + CONTEXT]
        truth = vals[:, s + CONTEXT : s + CONTEXT + HORIZON]
        cal = calendar_covariates(int(s), CONTEXT + HORIZON, 1, source.periods)
        res = fc.forecast(
            [ctx[i] for i in range(n)], HORIZON,
            future_covariates=[cal[i] for i in range(len(cal))],
        )
        below += (truth[..., None] <= res.quantiles).sum(axis=(0, 1))
        count += truth.size
    return below / count

emp = coverage(HOLDOUT)
emp_ft = coverage(HOLDOUT, fc=forecaster_ft)
nominal = np.arange(0.1, 1.0, 0.1)
plt.figure(figsize=(5, 4.5))
plt.plot([0, 1], [0, 1], "--", color="#999", label="perfect calibration")
plt.plot(nominal, emp, marker="o", label="zero-shot")
plt.plot(nominal, emp_ft, marker="s", label="fine-tuned")
plt.xlabel("nominal quantile level"); plt.ylabel("empirical coverage")
plt.title("ETTh2 quantile calibration")
plt.legend(); plt.tight_layout(); plt.show()
print(f"mean |coverage gap|  zero-shot: {np.mean(np.abs(emp-nominal)):.3f}   "
      f"fine-tuned: {np.mean(np.abs(emp_ft-nominal)):.3f}")

## A zero-shot forecast, qualitatively

In [ ]:
ch = 6  # OT (oil temperature), the usual ETT target
s = int(HOLDOUT.num_steps * 0.9)
ctx = HOLDOUT.values[:, s : s + CONTEXT]
truth = HOLDOUT.values[ch, s + CONTEXT : s + CONTEXT + HORIZON]
cal = calendar_covariates(s, CONTEXT + HORIZON, 1, HOLDOUT.periods)
fcov = [cal[i] for i in range(len(cal))]
res = forecaster.forecast([ctx[i] for i in range(len(ctx))], HORIZON, future_covariates=fcov)
res_ft = forecaster_ft.forecast([ctx[i] for i in range(len(ctx))], HORIZON, future_covariates=fcov)
x_ctx = np.arange(CONTEXT); x_hor = np.arange(CONTEXT, CONTEXT + HORIZON)
plt.figure(figsize=(11, 4))
plt.plot(x_ctx[-160:], ctx[ch, -160:], color="#444", lw=1.1, label="context")
plt.plot(x_hor, truth, ":", color="#444", lw=1.2, label="ground truth")
plt.plot(x_hor, res.point[ch], color="#d62728", lw=1.5, label="zero-shot point")
plt.plot(x_hor, res_ft.point[ch], color="#1f77b4", lw=1.5, label="fine-tuned point")
plt.fill_between(x_hor, res.quantiles[ch, :, 0], res.quantiles[ch, :, -1],
                 color="#d62728", alpha=0.13, label="zero-shot q10-q90")
plt.axvline(CONTEXT, color="#999", lw=0.8)
plt.title("ETTh2 'OT' — zero-shot vs fine-tuned (dataset never seen in pre-training)")
plt.legend(ncols=5, fontsize=9); plt.tight_layout(); plt.show()

In [ ]:
zs_nc = rows["ETTh2 (ZERO-SHOT dataset), no calendar"]
print("Takeaways")
print(f"- Zero-shot ETTh2 scaled MAE: model {zs['model']:.3f} vs seasonal-naive "
      f"{zs['seasonal-naive']:.3f} and last-value {zs['last-value']:.3f}")
print(f"- 600 fine-tuning steps on the target's own history: {zs['model']:.3f} -> {ft['model']:.3f} "
      f"({100*(zs['model']-ft['model'])/zs['model']:+.1f}%)")
print(f"- Calendar covariates (zero-shot): {zs_nc['model']:.3f} -> {zs['model']:.3f} "
      f"({100*(zs_nc['model']-zs['model'])/zs_nc['model']:+.1f}% from the known-future pathway)")
print(f"- Mean |quantile coverage gap|: {np.mean(np.abs(emp-nominal)):.3f} zero-shot, "
      f"{np.mean(np.abs(emp_ft-nominal)):.3f} fine-tuned")

## Limitations and next steps

- 5.2M parameters and 32M real points is still orders of magnitude below the released
  TimesFM-3 (334M params, >1T points); these results demonstrate a *productive
  pipeline*, not parity with the released model.
- The corpus, training and fine-tuning code all take the 334M `base()` config
  unchanged — the remaining knob is GPU hours.
- Practical recipes this notebook argues for: feed calendar structure through the
  past-future covariate pathway rather than hoping the context carries phase, and
  when you own any target history, a few hundred low-LR fine-tuning steps are close
  to free.